In [196]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_15452\3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [197]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [198]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv(r"C:\Users\Shrabani P\IronHack\Week4_Day3_NLP Embedding\Lab_Week4_Day3\lab-natural-language-processing\data\kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


In [199]:
print(data.columns)

Index(['text', 'label'], dtype='object')


In [200]:
data.head()

,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0


### Let's divide the training and test set into two partitions


In [201]:
# Your code
from sklearn.model_selection import train_test_split

X = data["text"]
y = data["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_val.shape)


(800,)
(200,)


In [202]:
test_data = pd.read_csv(
    r"C:\Users\Shrabani P\IronHack\Week4_Day3_NLP Embedding\Lab_Week4_Day3\lab-natural-language-processing\data\kg_test.csv",
    encoding="latin-1"
)

test_data.fillna("", inplace=True)

print(test_data.shape)

(5964, 1)


In [203]:
test_data.head()

,text
0,usiness is for the fact that the deceased man ...
1,They are happy to adjust to the afternoon. I a...
2,Lael Brainard was confirmed 78-19 this afterno...
3,H <hrod17@clintonemail.com>Friday March 26 201...
4,"n;""> Dear Good Friend,<br><br><br>I am happy t..."


## Data Preprocessing

In [204]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [205]:
import re

def clean_text(text):

    # Make sure the input is a string
    text = str(text)

    # Remove JavaScript

    text = re.sub(r'<script.*?>.*?</script>',
                  ' ',
                  text,
                  flags=re.DOTALL | re.IGNORECASE)

    # Remove CSS
    text = re.sub(r'<style.*?>.*?</style>',
                  ' ',
                  text,
                  flags=re.DOTALL | re.IGNORECASE)

    # Remove HTML comments
    
    text = re.sub(r'<!--.*?-->',
                  ' ',
                  text,
                  flags=re.DOTALL)

    # Remove HTML tags
    
    text = re.sub(r'<[^>]+>',
                  ' ',
                  text)

    
    # Remove special characters
    
    text = re.sub(r'[^a-zA-Z]',
                  ' ',
                  text)

    
    # Remove numbers
    
    text = re.sub(r'\d+',
                  ' ',
                  text)

    # Remove single characters

    text = re.sub(r'\b[a-zA-Z]\b',
                  ' ',
                  text)

    
    # Remove single character at beginning

    text = re.sub(r'^\s*[a-zA-Z]\s+',
                  '',
                  text)


    # Remove prefixed b
    
    text = re.sub(r'^b\s+',
                  '',
                  text)


    # Remove extra spaces

    text = re.sub(r'\s+',
                  ' ',
                  text)

    
    # Lowercase
    
    text = text.lower()

    return text.strip()

In [206]:
X_train = X_train.apply(clean_text)

X_val = X_val.apply(clean_text)

test_data["text"] = test_data["text"].apply(clean_text)

In [207]:
X_train.head()

442    dear good day hope fine cdear am writting this...
962    from mr henry kaborethe chief auditor incharge...
971                                              will do
190    from the desk of dr adamu ismalerauditing and ...
551    dear friend my name is loi estrada the wife of...
Name: text, dtype: object

In [208]:
test_data.head()

,text
0,usiness is for the fact that the deceased man ...
1,they are happy to adjust to the afternoon am g...
2,lael brainard was confirmed this afternoon mig...
3,friday march am sbwhoeop re have extended my c...
4,dear good friend am happy to inform you about ...


## Now let's work on removing stopwords
Remove the stopwords.

In [209]:
# Your code
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Shrabani
[nltk_data]     P\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [210]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

In [211]:
def remove_stopwords(text):
    words = text.split()

    filtered_words = [
        word for word in words
        if word not in stop_words
    ]

    return " ".join(filtered_words)

In [212]:
### apply on data
X_train = X_train.apply(remove_stopwords)

X_val = X_val.apply(remove_stopwords)

test_data["text"] = test_data["text"].apply(remove_stopwords)

In [213]:
X_train.head()

442    dear good day hope fine cdear writting mail du...
962    mr henry kaborethe chief auditor inchargeforei...
971                                                     
190    desk dr adamu ismalerauditing accounting manag...
551    dear friend name loi estrada wife mr josephest...
Name: text, dtype: object

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [214]:
# Your code
import nltk

nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to C:\Users\Shrabani
[nltk_data]     P\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Shrabani
[nltk_data]     P\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [215]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to C:\Users\Shrabani
[nltk_data]     P\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [216]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

In [217]:
def lemmatize_text(text):

    words = text.split()

    lemmas = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(lemmas)

In [218]:
X_train = X_train.apply(lemmatize_text)

X_val = X_val.apply(lemmatize_text)

test_data['text'] = test_data['text'].apply(lemmatize_text)

In [219]:
X_train.head()

442    dear good day hope fine cdear writting mail du...
962    mr henry kaborethe chief auditor inchargeforei...
971                                                     
190    desk dr adamu ismalerauditing accounting manag...
551    dear friend name loi estrada wife mr josephest...
Name: text, dtype: object

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [220]:
# Your code
ham = data[data["label"] == 0]
spam = data[data["label"] == 1]

In [221]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

# Ham
ham_bow = vectorizer.fit_transform(ham["text"])
ham_counts = ham_bow.toarray().sum(axis=0)

ham_words = pd.DataFrame({
    "Word": vectorizer.get_feature_names_out(),
    "Count": ham_counts
})

ham_top10 = ham_words.sort_values(by="Count", ascending=False).head(10)

print("Top 10 Ham Words")
print(ham_top10)

Top 10 Ham Words
      Word  Count
6819   the   1773
7021    to   1065
771    and    833
4834    of    791
3577    in    616
6799  that    414
3797    is    385
2959   for    369
4922    on    329
7712   you    311


In [222]:
spam_bow = vectorizer.fit_transform(spam["text"])
spam_counts = spam_bow.toarray().sum(axis=0)

spam_words = pd.DataFrame({
    "Word": vectorizer.get_feature_names_out(),
    "Count": spam_counts
})

spam_top10 = spam_words.sort_values(by="Count", ascending=False).head(10)

print("Top 10 Spam Words")
print(spam_top10)

Top 10 Spam Words
       Word  Count
17223   the   7046
17583    to   5593
13485    of   4984
4285    and   3985
10417    in   3289
20123   you   3229
17406  this   2675
12866    my   2143
20148  your   2078
8862    for   2030


## Extra features

In [223]:
data_train = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

data_val = pd.DataFrame({
    "text": X_val,
    "label": y_val
})

In [224]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,money_mark,suspicious_words,text_len
442,dear good day hope fine cdear writting mail du...,1,1,1,998
962,mr henry kaborethe chief auditor inchargeforei...,1,0,1,1946
971,,0,0,0,0
190,desk dr adamu ismalerauditing accounting manag...,1,1,1,383
551,dear friend name loi estrada wife mr josephest...,1,1,1,1475


## How would work the Bag of Words with Count Vectorizer concept?

### Bag of Words (BoW) is a technique for converting text into numerical features. In scikit-learn, the CountVectorizer class automatically creates the Bag of Words representation.

The basic idea is:

Collect all unique words from the training documents (this becomes the vocabulary).
Assign each unique word a column.
Count how many times each word appears in every document.
Create a matrix of these counts.

In [225]:
# Your code

## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [226]:
# Your code
from sklearn.feature_extraction.text import TfidfVectorizer

In [227]:
tfidf = TfidfVectorizer()

In [228]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

## And the Train a Classifier?

In [229]:
# Your code
X_train_tfidf = tfidf.fit_transform(data_train["text"])
X_val_tfidf = tfidf.transform(data_val["text"])

In [230]:
test_data.head()

,text
0,usiness fact deceased man foreigner authorized...
1,happy adjust afternoon going suggest pm start ...
2,lael brainard confirmed afternoon miguel rodri...
3,friday march sbwhoeop extended congrats
4,dear good friend happy inform succe ssin getti...


In [231]:
print(type(test_data))

<class 'pandas.core.frame.DataFrame'>


In [232]:
X_val_tfidf = tfidf.transform(data_val["text"])

X_test_tfidf = tfidf.transform(test_data["text"])


In [233]:
# Print shapes
print("Training shape:", X_train_tfidf.shape)
print("Validation shape:", X_val_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

Training shape: (800, 5000)
Validation shape: (200, 5000)
Test shape: (5964, 5000)


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [239]:
# Your code
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

In [240]:
### Bag of words only
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Vectorize
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(data_train["text"])
X_val_bow = bow.transform(data_val["text"])

# Train
model = MultinomialNB()
model.fit(X_train_bow, data_train["label"])

# Predict
y_pred = model.predict(X_val_bow)

print("Accuracy:", accuracy_score(data_val["label"], y_pred))

Accuracy: 0.98


In [241]:
### TF-IDF only
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(data_train["text"])
X_val_tfidf = tfidf.transform(data_val["text"])

model = MultinomialNB()
model.fit(X_train_tfidf, data_train["label"])

y_pred = model.predict(X_val_tfidf)

print("Accuracy:", accuracy_score(data_val["label"], y_pred))

Accuracy: 0.985


In [243]:
### Bag of Words + Additional Features
from scipy.sparse import hstack

extra_train = data_train[
    ["money_mark",
     "suspicious_words",
     "text_len"]
].values

extra_val = data_val[
    ["money_mark",
     "suspicious_words",
     "text_len"]
].values

X_train = hstack([X_train_bow, extra_train])

X_val = hstack([X_val_bow, extra_val])

model = MultinomialNB()

model.fit(X_train, data_train["label"])

y_pred = model.predict(X_val)

print("Accuracy:", accuracy_score(data_val["label"], y_pred))


Accuracy: 0.97


In [244]:
### TF-IDF + Additional Features
extra_train = data_train[
    ["money_mark",
     "suspicious_words",
     "text_len"]
].values

extra_val = data_val[
    ["money_mark",
     "suspicious_words",
     "text_len"]
].values

X_train = hstack([X_train_tfidf, extra_train])

X_val = hstack([X_val_tfidf, extra_val])

model = MultinomialNB()

model.fit(X_train, data_train["label"])

y_pred = model.predict(X_val)

print("Accuracy:", accuracy_score(data_val["label"], y_pred))

Accuracy: 0.905


#### The best-performing feature representations was: TF-IDF